# 21-03 · Движение корабля и враги

Практика к разделу [«Перемещаем корабль. Создаём и перемещаем врагов»](../../site/chapters/glava-21/21-03-dvizhenie-vragi.html).

## Цель

Ограничить движение корабля краями экрана и заставить врагов появляться и спускаться вниз.

## Про игровой цикл в этом ноутбуке

В обычном `.py`-файле игровой цикл — это `while rabotaet:`, зависящий от событий пользователя (закрытие окна, нажатия клавиш). В автоматически выполняемом ноутбуке некому создавать такие события, поэтому здесь мы прогоняем фиксированное число кадров через `for kadr in range(N):` — логика каждого отдельного кадра (движение, столкновения, отрисовка, `clock.tick()`) при этом точно такая же, как в настоящей игре из `projects/pygame/space-shooter/space_shooter.py`.

## Рабочий пример

In [1]:
import random

import pygame

SHIRINA, VYSOTA = 500, 600
FPS = 60

KORABL_SHIRINA, KORABL_VYSOTA = 50, 40
KORABL_SKOROST = 6

PULYA_SHIRINA, PULYA_VYSOTA = 4, 12
PULYA_SKOROST = 9

VRAG_SHIRINA, VRAG_VYSOTA = 40, 30
VRAG_SKOROST = 2
INTERVAL_POYAVLENIYA_VRAGA = 45

BELYJ = (255, 255, 255)
CHERNYJ = (10, 10, 20)
ZELYONYJ = (80, 220, 120)
KRASNYJ = (230, 60, 60)
ZHYOLTYJ = (240, 220, 80)

pygame.init()
screen = pygame.display.set_mode((SHIRINA, VYSOTA))
pygame.display.set_caption("Космический шутер")
clock = pygame.time.Clock()
shrift = pygame.font.SysFont(None, 32)
shrift_bolshoj = pygame.font.SysFont(None, 64)


def novaya_igra():
    return {
        "korabl": pygame.Rect(
            SHIRINA // 2 - KORABL_SHIRINA // 2,
            VYSOTA - KORABL_VYSOTA - 20,
            KORABL_SHIRINA,
            KORABL_VYSOTA,
        ),
        "puli": [],
        "vragi": [],
        "schet": 0,
        "kadrov_do_vraga": INTERVAL_POYAVLENIYA_VRAGA,
        "igra_okonchena": False,
    }


def obrabotat_klavishi(state, klavishi):
    korabl = state["korabl"]
    if klavishi[pygame.K_LEFT]:
        korabl.x -= KORABL_SKOROST
    if klavishi[pygame.K_RIGHT]:
        korabl.x += KORABL_SKOROST
    korabl.x = max(0, min(korabl.x, SHIRINA - KORABL_SHIRINA))


def vystrelit(state):
    korabl = state["korabl"]
    pulya = pygame.Rect(
        korabl.centerx - PULYA_SHIRINA // 2,
        korabl.top,
        PULYA_SHIRINA,
        PULYA_VYSOTA,
    )
    state["puli"].append(pulya)


def sozdat_vraga():
    x = random.randint(0, SHIRINA - VRAG_SHIRINA)
    return pygame.Rect(x, -VRAG_VYSOTA, VRAG_SHIRINA, VRAG_VYSOTA)


def obnovit_igru(state):
    if state["igra_okonchena"]:
        return

    for pulya in state["puli"]:
        pulya.y -= PULYA_SKOROST
    state["puli"] = [p for p in state["puli"] if p.bottom > 0]

    state["kadrov_do_vraga"] -= 1
    if state["kadrov_do_vraga"] <= 0:
        state["vragi"].append(sozdat_vraga())
        state["kadrov_do_vraga"] = INTERVAL_POYAVLENIYA_VRAGA

    for vrag in state["vragi"]:
        vrag.y += VRAG_SKOROST

    novye_puli = []
    novye_vragi = list(state["vragi"])
    for pulya in state["puli"]:
        popala = False
        for vrag in list(novye_vragi):
            if pulya.colliderect(vrag):
                novye_vragi.remove(vrag)
                state["schet"] += 10
                popala = True
                break
        if not popala:
            novye_puli.append(pulya)
    state["puli"] = novye_puli
    state["vragi"] = novye_vragi

    for vrag in state["vragi"]:
        if vrag.bottom >= VYSOTA or vrag.colliderect(state["korabl"]):
            state["igra_okonchena"] = True
            break


def narisovat(state):
    screen.fill(CHERNYJ)
    pygame.draw.rect(screen, ZELYONYJ, state["korabl"])
    for pulya in state["puli"]:
        pygame.draw.rect(screen, ZHYOLTYJ, pulya)
    for vrag in state["vragi"]:
        pygame.draw.rect(screen, KRASNYJ, vrag)

    tablo = shrift.render(f"Счёт: {state['schet']}", True, BELYJ)
    screen.blit(tablo, (10, 10))

    if state["igra_okonchena"]:
        nadpis = shrift_bolshoj.render("ИГРА ОКОНЧЕНА", True, BELYJ)
        rect = nadpis.get_rect(center=(SHIRINA // 2, VYSOTA // 2))
        screen.blit(nadpis, rect)

    pygame.display.flip()

pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.6)


In [2]:
state = novaya_igra()

# симулируем: стрелка вправо зажата 50 кадров подряд
for kadr in range(50):
    klavishi = {pygame.K_LEFT: False, pygame.K_RIGHT: True}
    obrabotat_klavishi(state, klavishi)
    obnovit_igru(state)
    narisovat(state)
    clock.tick(FPS)

print("Корабль после 50 кадров движения вправо:", state["korabl"])
print("Врагов появилось:", len(state["vragi"]))

Корабль после 50 кадров движения вправо: Rect(450, 540, 50, 40)
Врагов появилось: 1


## Проверка результата

In [3]:
assert state["korabl"].x == SHIRINA - KORABL_SHIRINA, "корабль должен упереться в правый край"
print("Верно: корабль остановился у правого края экрана, не выйдя за его пределы.")

assert len(state["vragi"]) >= 1, "за 50 кадров должен появиться хотя бы один враг (интервал 45)"
print(f"Найдено врагов: {len(state['vragi'])}")

Верно: корабль остановился у правого края экрана, не выйдя за его пределы.
Найдено врагов: 1


## Задание ★ Базовая практика

Прогоните ещё 45 кадров без движения корабля и убедитесь, что появился второй враг.

In [4]:
for kadr in range(45):
    klavishi = {pygame.K_LEFT: False, pygame.K_RIGHT: False}
    obrabotat_klavishi(state, klavishi)
    obnovit_igru(state)
    narisovat(state)

print("Врагов теперь:", len(state["vragi"]))
assert len(state["vragi"]) >= 2
print("Верно: появился второй враг.")

Врагов теперь: 2
Верно: появился второй враг.
